In [16]:
%matplotlib qt
import sys
from pathlib import Path

src_path = Path.cwd().parent
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

import time
import torch
import json
import gymnasium as gym
from envs.env import MiniGridEnvWrapper

from models.ppo_lstm import PPOLSTMAgent
from training.train import train_ppo_lstm
from training.evaluate import evaluate_agent

In [2]:
env_ids = [
    "MiniGrid-DoorKey-9x9-v0",
    "MiniGrid-SimpleCrossingS9N1-v0",
    "MiniGrid-LavaCrossingS9N2-v0",
    "MiniGrid-MultiRoom-N2-S4-v0",
    "MiniGrid-KeyCorridorS3R3-v0"
]
env = MiniGridEnvWrapper(env_ids[1], render_mode='rgb_array')

obs, info = env.reset()
print(f"Initial observation shape: {obs.shape}")
print(f"Action space: {env.action_space}")

num_steps = 100
for step in range(num_steps):
    action = env.action_space.sample()
    
    obs, reward, terminated, truncated, info = env.step(action)
    
    print(f"Step {step + 1}: Action={action}, Reward={reward}, Done={terminated or truncated}")
    
    env.display_interactive()
    
    if terminated or truncated:
        print(f"Episode finished after {step + 1} steps!")
        obs, info = env.reset()
        print("Environment reset for next episode")
        env.display_interactive()
    time.sleep(0.01)
env.close()

Initial observation shape: (7, 7, 3)
Action space: Discrete(7)
Step 1: Action=3, Reward=0, Done=False
Step 2: Action=1, Reward=0, Done=False
Step 3: Action=1, Reward=0, Done=False
Step 4: Action=0, Reward=0, Done=False
Step 5: Action=1, Reward=0, Done=False
Step 6: Action=6, Reward=0, Done=False
Step 7: Action=3, Reward=0, Done=False
Step 8: Action=6, Reward=0, Done=False
Step 9: Action=5, Reward=0, Done=False
Step 10: Action=6, Reward=0, Done=False
Step 11: Action=4, Reward=0, Done=False
Step 12: Action=2, Reward=0, Done=False
Step 13: Action=6, Reward=0, Done=False
Step 14: Action=1, Reward=0, Done=False
Step 15: Action=5, Reward=0, Done=False
Step 16: Action=5, Reward=0, Done=False
Step 17: Action=5, Reward=0, Done=False
Step 18: Action=1, Reward=0, Done=False
Step 19: Action=5, Reward=0, Done=False
Step 20: Action=2, Reward=0, Done=False
Step 21: Action=0, Reward=0, Done=False
Step 22: Action=3, Reward=0, Done=False
Step 23: Action=3, Reward=0, Done=False
Step 24: Action=0, Reward=

In [5]:
agent = train_ppo_lstm(
        env=env,
        experiment_name="ppo_lstm_simple_crossing_baseline",
        num_iterations=300,
        print_interval=1,
        steps_per_iteration=2048,
        save_interval=100,
        device="cpu",  # or "cuda" or "mps"
        lr=3e-4,
        gamma=0.99,
        ppo_epochs=4,
        ppo_minibatch_size=4,
        hidden_size=256,
    )

Logging to: ../runs/ppo_lstm_simple_crossing_baseline_20251110_141813

Starting training: ppo_lstm_simple_crossing_baseline_20251110_141813

[   1/300] Frames:   2,048 | Reward:    0.00 | Length:  324.0 | Loss: 0.0103/0.0000
[   2/300] Frames:   4,096 | Reward:    0.11 | Length:  291.0 | Loss: -0.0298/0.0027
[   3/300] Frames:   6,144 | Reward:    0.10 | Length:  300.2 | Loss: -0.0134/0.0007
[   4/300] Frames:   8,192 | Reward:    0.00 | Length:  324.0 | Loss: 0.0012/0.0001
[   5/300] Frames:  10,240 | Reward:    0.05 | Length:  311.5 | Loss: 0.0126/0.0003
[   6/300] Frames:  12,288 | Reward:    0.00 | Length:  324.0 | Loss: 0.0193/0.0000
[   7/300] Frames:  14,336 | Reward:    0.09 | Length:  298.2 | Loss: -0.0417/0.0015
[   8/300] Frames:  16,384 | Reward:    0.07 | Length:  304.0 | Loss: -0.0239/0.0010
[   9/300] Frames:  18,432 | Reward:    0.06 | Length:  307.5 | Loss: -0.0069/0.0007
[  10/300] Frames:  20,480 | Reward:    0.00 | Length:  324.0 | Loss: 0.0060/0.0000
[  11/300] Fra

In [19]:
agent_class = PPOLSTMAgent
env = MiniGridEnvWrapper(env_ids[1], render_mode='rgb_array')
config_path = "../runs/ppo_lstm_simple_crossing_baseline_20251110_141813/config.json"
checkpoint_path = "../checkpoints/ppo_lstm_simple_crossing_baseline_20251110_141813/final_model.pt"

results = evaluate_agent(agent_class, env, checkpoint_path, config_path)

Loading checkpoint from ../checkpoints/ppo_lstm_simple_crossing_baseline_20251110_141813/final_model.pt...
Checkpoint loaded successfully!

Running 10 evaluation episodes...
Episode  1/10 | Steps:  21 | Reward:   0.94
Episode  2/10 | Steps:  17 | Reward:   0.95
Episode  3/10 | Steps:  14 | Reward:   0.96
Episode  4/10 | Steps:  14 | Reward:   0.96
Episode  5/10 | Steps:  17 | Reward:   0.95
Episode  6/10 | Steps:  38 | Reward:   0.89
Episode  7/10 | Steps:  17 | Reward:   0.95
Episode  8/10 | Steps:  13 | Reward:   0.96
Episode  9/10 | Steps:  14 | Reward:   0.96
Episode 10/10 | Steps:  14 | Reward:   0.96

Evaluation Summary:
  Episodes:     10
  Mean Reward:  0.95 ± 0.02
  Min/Max:      0.89 / 0.96
  Mean Length:  17.90 ± 7.08
  Success Rate: 100.0%
